# 02 — Data Cleaning Pipeline

**Dataset:** Inside Airbnb — New York City 2019 (`AB_NYC_2019.csv`)  
**Output:** `data/processed/airbnb_nyc_cleaned.csv` — standardised, enriched, dashboard-ready.

**Design Philosophy:**  
This notebook orchestrates the production cleaning pipeline. Every transformation is preceded by a *why* (business or statistical rationale) and followed by a *verification* check. We leverage the core functions in `scripts/etl_pipeline.py` to ensure consistency between our interactive work and production runs.

**Topic Coverage:**
| Topic Group | Applied In |
|---|---|
| NumPy arrays, masking, vectorized ops | §2.2 – §2.4 |
| Pandas loading, inspection, missing values, type conversion, duplicate handling | §2.1 |
| Descriptive stats, distribution checks, outliers | §2.3, §2.5 |

---
## 2.0 Environment & Pipeline Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import (
    normalize_columns, rename_verbose_columns, drop_duplicates,
    convert_types, fill_missing, treat_price_outliers, 
    treat_min_nights, engineer_features, FINAL_COLUMNS
)

RAW_PATH      = PROJECT_ROOT / 'data' / 'raw'       / 'AB_NYC_2019.csv'
CLEANED_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'

class TransformLog:
    """Lightweight auditor that records shape & null changes after each pipeline step."""
    def __init__(self):
        self._entries = []
    def record(self, step: str, df: pd.DataFrame, note: str = '') -> None:
        self._entries.append({
            'step'       : step,
            'rows'       : len(df),
            'cols'       : len(df.columns),
            'total_nulls': int(df.isna().sum().sum()),
            'note'       : note,
        })
    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self._entries)

log = TransformLog()

---
## 2.1 Load & Standardise

**Why:** Initial load must be profiled for base null rates. Standardisation ensures consistent field access.

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
log.record('01_load_raw', df, 'initial state')

df = normalize_columns(df)
df = rename_verbose_columns(df)
log.record('02_standardize', df, 'snake_case + kpi renames')
display(df.head(2))

---
## 2.2 Integrity Phase: Duplicates & Types

In [ ]:
df = drop_duplicates(df)
df = convert_types(df)
log.record('03_integrity', df, 'duplicates handled, types cast')
print(f"Unique Listings: {df['id'].nunique():,}")

---
## 2.3 Quality Phase: Nulls & Outliers

In [ ]:
df = fill_missing(df)
df = treat_price_outliers(df)
df = treat_min_nights(df)
log.record('04_quality', df, 'nulls imputed, outliers capped')
print(f"Capped Price Range: ${df['price'].min()} - ${df['price'].max()}")

---
## 2.4 Enrichment Phase: Feature Engineering

In [ ]:
df = engineer_features(df)
log.record('05_enrichment', df, 'kpis and segments added')
display(df[['price', 'revenue_proxy', 'occupancy_rate_est', 'price_tier']].head())

---
## 2.5 Audit & Export

In [ ]:
df_clean = df[[c for c in FINAL_COLUMNS if c in df.columns]].copy()
df_clean.to_csv(CLEANED_PATH, index=False)
log.record('06_export', df_clean, 'final schema saved')

audit_df = log.to_dataframe()
audit_df['row_delta'] = audit_df['rows'].diff().fillna(0)
audit_df